
# OMNet-V3 — Final Implementation Notebook

This notebook implements the approved OMNet-V3 design for BreaKHis classification using a dual-branch EfficientNet-B0 + ViT-Tiny fusion model with magnification-aware gating, patient-disjoint 5-fold validation, and the locked loss/training configuration.


In [ ]:
CONFIG = {
    "seed": 42,
    "device": "cuda" if __import__('torch').cuda.is_available() else "cpu",
    "output_dir": "/content/drive/MyDrive/output_v3",
    "dataset_root": "/content/drive/MyDrive/BreakHis/BreaKHis_v1/histology_slides/breast",
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 4,
    "max_epochs": 50,
    "n_splits": 5,
    "early_stopping_patience": 10,
    "base_lr": 1e-4,
    "head_lr": 5e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "gradient_clip_norm": 1.0,
    "stain_aug_prob": 0.5,
    "dropout": 0.3,
    "fusion_dim": 256,
    "magnification_embedding_dim": 64,
    "loss_weights": {"binary": 0.3, "subtype": 0.6, "consistency": 0.1},
    "magnification_levels": [40, 100, 200, 400],
    "binary_classes": ["benign", "malignant"],
    "subtype_order": ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"],
    "subtype_to_index": {"A": 0, "F": 1, "PT": 2, "TA": 3, "DC": 4, "LC": 5, "MC": 6, "PC": 7},
    "binary_mapping": {"A": 0, "F": 0, "PT": 0, "TA": 0, "DC": 1, "LC": 1, "MC": 1, "PC": 1},
    "train_resize": 256,
    "train_crop": 224,
    "val_resize": 256,
    "val_crop": 224,
    "amp_enabled": True,
    "progressive_unfreeze_start_epoch": 6,
    "use_drive_mount": True,
    "metadata_columns": [
        "file_path",
        "filename",
        "class_name",
        "subtype",
        "subtype_index",
        "binary_label",
        "magnification",
        "magnification_index",
        "patient_id",
        "sequence",
    ],
}
print(f"Config updated. New Dataset root: {CONFIG['dataset_root']}")

Config updated. New Dataset root: /content/drive/MyDrive/BreakHis/BreaKHis_v1/histology_slides/breast



## 1. Configuration

All hyperparameters, paths, and lock decisions are centralized in the `CONFIG` dictionary.


In [ ]:

%pip install -q timm>=0.9.0

import os
import json
import random
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import functional as TF
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from sklearn.manifold import TSNE
from tqdm.auto import tqdm
import timm

warnings.filterwarnings("ignore")

np.random.seed(CONFIG["seed"])
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print(f"PyTorch version: {torch.__version__}")
print(f"timm version: {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.11.0+cu128
timm version: 1.0.28
CUDA available: True
GPU: Tesla T4



## 2. Environment and GPU Setup


In [ ]:

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True


Using device: cuda


In [ ]:
# ============================================================
# Section 3: Mount Google Drive & Validate Storage
# ============================================================
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('.')

# Updated to match user's actual folder names
DATASET_ROOT = Path(CONFIG['dataset_root'])
OUTPUT_DIR = Path(CONFIG['output_dir'])
PROJECT_ROOT = OUTPUT_DIR

# --- Output sub-directories ---
GRADCAM_DIR = OUTPUT_DIR / 'gradcam'
SCORECAM_DIR = OUTPUT_DIR / 'scorecam'
MISCLASSIFIED_DIR = OUTPUT_DIR / 'misclassified'
CORRECT_PRED_DIR = OUTPUT_DIR / 'correct_predictions'

print("=" * 70)
print("Google Drive & Storage Validation")
print("=" * 70)
print(f"Drive root exists    : {DRIVE_ROOT.exists()}")
print(f"Dataset path         : {DATASET_ROOT}")
print(f"Dataset exists       : {DATASET_ROOT.exists()}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATASET_ROOT}")

# --- FIX: Debug Dataset Contents ---
print("\nContents of Dataset Directory:")
try:
    for item in DATASET_ROOT.iterdir():
        print(f" - {item.name}")
except Exception as e:
    print(f"Could not read contents: {e}")

# Check for expected BreaKHis folders instead of throwing a strict error on 'SOB'
if (DATASET_ROOT / 'BreaKHis_v1').exists():
    print("\nINFO: Found 'BreaKHis_v1'. You may need to update CONFIG['dataset_root'] to point deeper (e.g., /BreaKHis_v1/histology_slides/breast/).")
elif (DATASET_ROOT / 'benign').exists() and (DATASET_ROOT / 'malignant').exists():
    print("\nINFO: Found 'benign' and 'malignant' folders. Root path is correct.")
else:
    print("\nWARNING: Standard subdirectories not found at this root. Check the printed contents above.")

# Create output structure
for d in [OUTPUT_DIR, GRADCAM_DIR, SCORECAM_DIR, MISCLASSIFIED_DIR, CORRECT_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("VALIDATION COMPLETE: Output directories prepared.")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive & Storage Validation
Drive root exists    : True
Dataset path         : /content/drive/MyDrive/BreakHis/BreaKHis_v1/histology_slides/breast
Dataset exists       : True

Contents of Dataset Directory:
 - README.txt
 - count_files.sh
 - malignant
 - benign

INFO: Found 'benign' and 'malignant' folders. Root path is correct.

VALIDATION COMPLETE: Output directories prepared.



## 3. Dataset Discovery

The dataset will be discovered recursively by scanning the drive root and parsing the metadata embedded in file names.


In [ ]:
def parse_breakhis_filename(path: str) -> Optional[dict]:
    filename = os.path.basename(path)
    if not filename.lower().endswith('.png'):
        return None

    # Example: SOB_B_A-14-22549AB-40-001.png
    name = filename.split('.png')[0]
    parts = name.split('-')

    # The standard convention has 6 parts separated by hyphens after the SOB_X_X prefix
    # Parts: [SOB_B_A, 14, 22549AB, 40, 001]
    if len(parts) < 4:
        return None

    try:
        # Prefix part contains Class and Subtype
        prefix_parts = parts[0].split('_') # [SOB, B, A]
        if len(prefix_parts) < 3:
            return None

        raw_class = prefix_parts[1]
        raw_subtype = prefix_parts[2]

        # Magnification is usually the second to last part
        mag_token = parts[-2]
        magnification = int(mag_token)

        if magnification not in CONFIG['magnification_levels']:
            return None

        class_name = 'benign' if raw_class == 'B' else 'malignant'

        # Map subtype token to our standard labels
        subtype = raw_subtype
        # Some datasets use DC for Ductal Carcinoma, etc.
        if subtype not in CONFIG['subtype_to_index']:
            return None

        patient_id = f"{parts[1]}-{parts[2]}" # e.g. 14-22549AB
        seq = int(parts[-1])

        return {
            'file_path': path,
            'filename': filename,
            'class_name': class_name,
            'subtype': subtype,
            'subtype_index': CONFIG['subtype_to_index'][subtype],
            'binary_label': CONFIG['binary_mapping'][subtype],
            'magnification': magnification,
            'magnification_index': CONFIG['magnification_levels'].index(magnification),
            'patient_id': patient_id,
            'sequence': seq,
        }
    except (ValueError, IndexError):
        return None

def discover_breakhis_dataset(dataset_root: str) -> pd.DataFrame:
    records = []
    path_obj = Path(dataset_root)
    if not path_obj.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    print(f"Scanning {dataset_root} for PNG files...")
    for full_path in path_obj.rglob('*.png'):
        parsed = parse_breakhis_filename(str(full_path))
        if parsed is not None:
            records.append(parsed)

    if len(records) == 0:
        raise RuntimeError(f"No valid BreaKHis images found. Check if {dataset_root} contains PNGs following the SOB_... convention.")

    df = pd.DataFrame(records)
    return df

metadata = discover_breakhis_dataset(CONFIG['dataset_root'])
print(f"\nSuccess! Discovered {len(metadata)} images.")
print(f"Unique patients: {metadata['patient_id'].nunique()}")
display(metadata.head())

Scanning /content/drive/MyDrive/BreakHis/BreaKHis_v1/histology_slides/breast for PNG files...

Success! Discovered 7909 images.
Unique patients: 81


,file_path,filename,class_name,subtype,subtype_index,binary_label,magnification,magnification_index,patient_id,sequence
0,/content/drive/MyDrive/BreakHis/BreaKHis_v1/hi...,SOB_M_MC-14-13413-200-003.png,malignant,MC,6,1,200,2,14-13413,3
1,/content/drive/MyDrive/BreakHis/BreaKHis_v1/hi...,SOB_M_MC-14-13413-200-004.png,malignant,MC,6,1,200,2,14-13413,4
2,/content/drive/MyDrive/BreakHis/BreaKHis_v1/hi...,SOB_M_MC-14-13413-200-001.png,malignant,MC,6,1,200,2,14-13413,1
3,/content/drive/MyDrive/BreakHis/BreaKHis_v1/hi...,SOB_M_MC-14-13413-200-002.png,malignant,MC,6,1,200,2,14-13413,2
4,/content/drive/MyDrive/BreakHis/BreaKHis_v1/hi...,SOB_M_MC-14-13413-200-006.png,malignant,MC,6,1,200,2,14-13413,6


In [ ]:
print('By class:')
print(metadata['class_name'].value_counts().sort_index())
print('\nBy subtype:')
print(metadata['subtype'].value_counts().sort_index())
print('\nBy magnification:')
print(metadata['magnification'].value_counts().sort_index())
print('\nPatients per subtype:')
print(metadata.groupby('subtype')['patient_id'].nunique().sort_index())

By class:
class_name
benign       2480
malignant    5429
Name: count, dtype: int64

By subtype:
subtype
A      444
DC    3451
F     1014
LC     626
MC     792
PC     560
PT     453
TA     569
Name: count, dtype: int64

By magnification:
magnification
40     1995
100    2081
200    2013
400    1820
Name: count, dtype: int64

Patients per subtype:
subtype
A      4
DC    38
F     10
LC     5
MC     9
PC     6
PT     3
TA     7
Name: patient_id, dtype: int64



## 4. Patient-Disjoint Splitting

The data split must be patient-disjoint and preserve all magnifications belonging to the same patient in the same fold.


In [ ]:

X = metadata[['file_path', 'subtype', 'patient_id']].copy()
y = metadata['subtype'].values
groups = metadata['patient_id'].values

sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
split_indices = []
for train_idx, test_idx in sgkf.split(X, y, groups):
    split_indices.append((train_idx, test_idx))

print(f'Created {len(split_indices)} folds.')
for fold_idx, (train_idx, test_idx) in enumerate(split_indices):
    train_df = metadata.iloc[train_idx].copy()
    test_df = metadata.iloc[test_idx].copy()
    print(f'Fold {fold_idx}: train={len(train_df)} | test={len(test_df)}')
    print(f"  train patients: {train_df['patient_id'].nunique()} | test patients: {test_df['patient_id'].nunique()}")


Created 5 folds.
Fold 0: train=6265 | test=1644
  train patients: 64 | test patients: 17
Fold 1: train=6638 | test=1271
  train patients: 67 | test patients: 14
Fold 2: train=6361 | test=1548
  train patients: 66 | test patients: 15
Fold 3: train=6331 | test=1578
  train patients: 64 | test patients: 17
Fold 4: train=6041 | test=1868
  train patients: 63 | test patients: 18


In [ ]:

def build_fold_split(fold_idx: int, metadata_df: pd.DataFrame):
    X = metadata_df[['file_path', 'subtype', 'patient_id']].copy()
    y = metadata_df['subtype'].values
    groups = metadata_df['patient_id'].values

    sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
    folds = list(sgkf.split(X, y, groups))
    train_idx, test_idx = folds[fold_idx]

    train_df = metadata_df.iloc[train_idx].copy().reset_index(drop=True)
    test_df = metadata_df.iloc[test_idx].copy().reset_index(drop=True)
    return train_df, test_df

fold_train_df, fold_test_df = build_fold_split(0, metadata)
print(f"Fold 0 train shape={fold_train_df.shape}, test shape={fold_test_df.shape}")


Fold 0 train shape=(6265, 10), test shape=(1644, 10)


In [ ]:

# This cell is intentionally the explicit stain augmentation implementation specified in the locked design.
def rgb_to_od(img):
    img = img.astype(np.float32) / 255.0
    eps = 1e-6
    img = np.clip(img, eps, 1.0)
    return -np.log(img)

def estimate_he_vectors(od):
    # Macenko-inspired SVD estimate with a conservative numerical implementation.
    od = od.reshape(-1, 3)
    od = od[~np.any(np.isinf(od) | np.isnan(od), axis=1)]
    if od.shape[0] == 0:
        return np.eye(3)
    _, _, vh = np.linalg.svd(od, full_matrices=False)
    return vh[:2, :]

def macenko_perturb(img: np.ndarray, p: float = 0.5):
    if np.random.rand() > p:
        return img

    arr = np.asarray(img)
    if arr.ndim != 3 or arr.shape[-1] != 3:
        return img

    od = rgb_to_od(arr)
    he_basis = estimate_he_vectors(od)
    stain = np.dot(od.reshape(-1, 3), he_basis.T)
    stain = stain.reshape(arr.shape[0], arr.shape[1], 2)
    if stain.size == 0:
        return img

    rand_factors = np.random.uniform(0.85, 1.15, size=(2,))
    rand_bias = np.random.uniform(-0.05, 0.05, size=(2,))
    perturbed = stain * rand_factors + rand_bias
    recon = np.dot(perturbed.reshape(-1, 2), he_basis).reshape(arr.shape[0], arr.shape[1], 3)
    recon = np.clip(recon, 0.0, 1.0)
    return (recon * 255.0).astype(np.uint8)

def color_jitter_fallback(img):
    jitter = transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)
    return np.array(jitter(Image.fromarray(img)))

try:
    from PIL import Image
    print('Stain augmentation ready: Macenko perturbation enabled.')
except Exception:
    print('Pillow not available; using fallback path only.')


Stain augmentation ready: Macenko perturbation enabled.



## 5. Dataset Class and DataLoader Setup


In [ ]:
from PIL import Image

class BreakHisDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split: str = 'train', transform=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)
        # The transforms (build_transforms or custom_train_transform) already convert the PIL Image
        # to a torch.Tensor and handle normalization, so no further processing is needed here.

        return {
            'image': image,
            'binary_label': int(row['binary_label']),
            'subtype_label': int(row['subtype_index']),
            'magnification_index': int(row['magnification_index']),
            'patient_id': row['patient_id'],
            'file_path': row['file_path'],
        }

def build_transforms(split: str):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize'])),
            transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(270, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    return transforms.Compose([
        transforms.Resize((CONFIG['val_resize'], CONFIG['val_resize'])),
        transforms.CenterCrop(CONFIG['val_crop']),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

def custom_train_transform(image: Image.Image):
    image = transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize']))(image)
    image = transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0))(image)
    image = transforms.RandomHorizontalFlip(p=0.5)(image)
    image = transforms.RandomVerticalFlip(p=0.5)(image)
    image = transforms.RandomRotation(270, interpolation=transforms.InterpolationMode.BILINEAR)(image)

    if np.random.rand() < CONFIG['stain_aug_prob']:
        image_np = np.array(image)
        image_np = macenko_perturb(image_np, p=1.0)
        image = Image.fromarray(image_np.astype(np.uint8))

    image = transforms.ToTensor()(image)
    image = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))(image)
    return image

def build_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_ds = BreakHisDataset(train_df, 'train', custom_train_transform)
    val_ds = BreakHisDataset(val_df, 'val', build_transforms('val'))
    test_ds = BreakHisDataset(test_df, 'test', build_transforms('val'))

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'])
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    return train_loader, val_loader, test_loader

print('Dataset and loader helpers are ready.')

Dataset and loader helpers are ready.


In [ ]:

class MagnificationAwareFusion(nn.Module):
    def __init__(self, feat_dim=256, mag_dim=64):
        super().__init__()
        self.magnification_embedding = nn.Embedding(4, mag_dim)
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2 + mag_dim, 1),
            nn.Sigmoid(),
        )
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim + mag_dim, feat_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, f_cnn, f_vit, magnification_index):
        e_m = self.magnification_embedding(magnification_index)
        concat = torch.cat([f_cnn, f_vit, e_m], dim=-1)
        alpha = self.gate(concat)
        fused = alpha * f_cnn + (1.0 - alpha) * f_vit
        residual = self.mlp(torch.cat([fused, e_m], dim=-1))
        out = fused + residual
        return out, alpha

class OMNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn_backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.vit_backbone = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0)

        self.cnn_proj = nn.Sequential(
            nn.Linear(1280, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(192, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.fusion = MagnificationAwareFusion(feat_dim=CONFIG['fusion_dim'], mag_dim=CONFIG['magnification_embedding_dim'])
        self.binary_head = nn.Linear(CONFIG['fusion_dim'], 2)
        self.subtype_head = nn.Linear(CONFIG['fusion_dim'], 8)

    def forward_cnn(self, x):
        x = self.cnn_backbone.forward_features(x)
        x = self.cnn_backbone.global_pool(x)
        if x.dim() == 4:
            x = x.flatten(1)
        return self.cnn_proj(x)

    def forward_vit(self, x):
        x = self.vit_backbone.forward_features(x)
        if isinstance(x, (tuple, list)):
            x = x[0]
        if x.dim() == 3:
            x = x[:, 0, :]
        return self.vit_proj(x)

    def forward(self, x, magnification_index):
        f_cnn = self.forward_cnn(x)
        f_vit = self.forward_vit(x)
        f_out, alpha = self.fusion(f_cnn, f_vit, magnification_index)
        binary_logits = self.binary_head(f_out)
        subtype_logits = self.subtype_head(f_out)
        return {
            'f_out': f_out,
            'alpha': alpha,
            'binary_logits': binary_logits,
            'subtype_logits': subtype_logits,
        }

model = OMNetV3().to(device)
print(model)


OMNetV3(
  (cnn_backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): C


## 6. Loss Function


In [ ]:

class HierarchicalLoss(nn.Module):
    def __init__(self, class_weights_binary: torch.Tensor, class_weights_subtype: torch.Tensor):
        super().__init__()
        self.loss_binary = nn.CrossEntropyLoss(weight=class_weights_binary)
        self.loss_subtype = nn.CrossEntropyLoss(weight=class_weights_subtype)

    def forward(self, binary_logits, subtype_logits, binary_targets, subtype_targets):
        L_binary = self.loss_binary(binary_logits, binary_targets)
        L_subtype = self.loss_subtype(subtype_logits, subtype_targets)

        subtype_probs = torch.softmax(subtype_logits, dim=1)
        grouped = torch.stack([
            subtype_probs[:, :4].sum(dim=1),
            subtype_probs[:, 4:8].sum(dim=1),
        ], dim=1)
        consistency_target = torch.softmax(binary_logits, dim=1)
        L_consistency = F.kl_div(
            torch.log_softmax(grouped, dim=1),
            consistency_target,
            reduction='batchmean',
            log_target=False,
        )

        total = 0.3 * L_binary + 0.6 * L_subtype + 0.1 * L_consistency
        return total, {'binary': L_binary, 'subtype': L_subtype, 'consistency': L_consistency}

def compute_class_weights(train_df: pd.DataFrame):
    n_total = len(train_df)
    binary_counts = train_df['binary_label'].value_counts().sort_index()
    subtype_counts = train_df['subtype_index'].value_counts().sort_index()

    binary_weights = torch.tensor([
        n_total / (2 * binary_counts.get(i, 1)) for i in range(2)
    ], dtype=torch.float32)
    subtype_weights = torch.tensor([
        n_total / (8 * subtype_counts.get(i, 1)) for i in range(8)
    ], dtype=torch.float32)
    return binary_weights, subtype_weights

print('Loss and weighting helpers are ready.')


Loss and weighting helpers are ready.



## 7. Training Engine


In [ ]:
def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad = flag

def train_one_epoch(model, loader, optimizer, scaler, criterion, device):
    model.train()
    running_loss = 0.0
    preds_binary = []
    preds_subtype = []
    labels_binary = []
    labels_subtype = []

    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device)
        binary_targets = batch['binary_label'].to(device)
        subtype_targets = batch['subtype_label'].to(device)
        magnification_idx = batch['magnification_index'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=CONFIG['amp_enabled']):
            outputs = model(images, magnification_idx)
            loss, loss_terms = criterion(
                outputs['binary_logits'],
                outputs['subtype_logits'],
                binary_targets,
                subtype_targets,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds_binary.append(torch.argmax(outputs['binary_logits'], dim=1).cpu())
        preds_subtype.append(torch.argmax(outputs['subtype_logits'], dim=1).cpu())
        labels_binary.append(binary_targets.cpu())
        labels_subtype.append(subtype_targets.cpu())

    epoch_loss = running_loss / len(loader.dataset)
    binary_pred = torch.cat(preds_binary)
    subtype_pred = torch.cat(preds_subtype)
    binary_true = torch.cat(labels_binary)
    subtype_true = torch.cat(labels_subtype)

    metrics = {
        'loss': epoch_loss,
        'binary_accuracy': accuracy_score(binary_true.numpy(), binary_pred.numpy()),
        'subtype_accuracy': accuracy_score(subtype_true.numpy(), subtype_pred.numpy()),
        'subtype_macro_f1': f1_score(subtype_true.numpy(), subtype_pred.numpy(), average='macro'),
    }
    return metrics

def evaluate_model(model, loader, device):
    model.eval()
    logits_binary = []
    logits_subtype = []
    labels_binary = []
    labels_subtype = []
    patient_ids = []
    files = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            binary_targets = batch['binary_label']
            subtype_targets = batch['subtype_label']
            magnification_idx = batch['magnification_index'].to(device)

            with torch.autocast(device_type=device.type, enabled=CONFIG['amp_enabled']):
                outputs = model(images, magnification_idx)

            logits_binary.append(outputs['binary_logits'].cpu())
            logits_subtype.append(outputs['subtype_logits'].cpu())
            labels_binary.append(binary_targets)
            labels_subtype.append(subtype_targets)
            patient_ids.extend(batch['patient_id'])
            files.extend(batch['file_path'])

    binary_probs = torch.softmax(torch.cat(logits_binary), dim=1).numpy()
    subtype_probs = torch.softmax(torch.cat(logits_subtype), dim=1).numpy()
    binary_preds = np.argmax(binary_probs, axis=1)
    subtype_preds = np.argmax(subtype_probs, axis=1)
    binary_true = torch.cat(labels_binary).numpy()
    subtype_true = torch.cat(labels_subtype).numpy()

    metrics = {
        'binary_accuracy': accuracy_score(binary_true, binary_preds),
        'binary_macro_f1': f1_score(binary_true, binary_preds, average='macro'),
        'binary_balanced_accuracy': balanced_accuracy_score(binary_true, binary_preds),
        'binary_mcc': matthews_corrcoef(binary_true, binary_preds),
        'subtype_accuracy': accuracy_score(subtype_true, subtype_preds),
        'subtype_macro_f1': f1_score(subtype_true, subtype_preds, average='macro'),
        'subtype_weighted_f1': f1_score(subtype_true, subtype_preds, average='weighted'),
        'subtype_balanced_accuracy': balanced_accuracy_score(subtype_true, subtype_preds),
        'subtype_mcc': matthews_corrcoef(subtype_true, subtype_preds),
        'binary_confusion_matrix': confusion_matrix(binary_true, binary_preds),
        'subtype_confusion_matrix': confusion_matrix(subtype_true, subtype_preds),
        'subtype_probs': subtype_probs,
        'binary_probs': binary_probs,
        'subtype_preds': subtype_preds,
        'binary_preds': binary_preds,
        'subtype_true': subtype_true, # Added true labels
        'binary_true': binary_true,   # Added true labels
        'patient_ids': patient_ids,
        'files': files,
    }
    return metrics

print('Training and evaluation functions are ready.')

Training and evaluation functions are ready.


In [ ]:
def run_fold(fold_idx: int, full_df: pd.DataFrame):
    # Create a specific output directory for this fold
    fold_output_dir = OUTPUT_DIR / f'fold_{fold_idx}'
    fold_output_dir.mkdir(parents=True, exist_ok=True)

    train_df, test_df = build_fold_split(fold_idx, full_df)
    train_patients = set(train_df['patient_id'])
    remaining = full_df[~full_df['patient_id'].isin(train_patients)].copy()
    val_patients = remaining['patient_id'].drop_duplicates().sample(max(1, int(0.1 * len(train_patients))), random_state=CONFIG['seed'])
    val_df = full_df[full_df['patient_id'].isin(val_patients)].copy()
    train_df = train_df[~train_df['patient_id'].isin(val_patients)].copy()

    binary_weights, subtype_weights = compute_class_weights(train_df)
    binary_weights = binary_weights.to(device)
    subtype_weights = subtype_weights.to(device)
    criterion = HierarchicalLoss(binary_weights, subtype_weights).to(device)

    train_loader, val_loader, test_loader = build_dataloaders(train_df, val_df, test_df)

    model = OMNetV3().to(device)

    # Correctly define optimizer groups to avoid overlap
    base_params = []
    head_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        # Parameters for backbones (should get base_lr)
        if 'cnn_backbone' in name or 'vit_backbone' in name:
            base_params.append(p)
        # Parameters for projection layers, fusion block, and classification heads (should get head_lr)
        elif 'proj' in name or 'fusion' in name or 'binary_head' in name or 'subtype_head' in name:
            head_params.append(p)
        # It's good practice to ensure all trainable parameters are assigned
        else:
            print(f"WARNING: Parameter '{name}' not explicitly assigned to an optimizer group.")

    optimizer_groups = [
        {'params': base_params, 'lr': CONFIG['base_lr']},
        {'params': head_params, 'lr': CONFIG['head_lr']},
    ]

    optimizer = torch.optim.AdamW(optimizer_groups, weight_decay=CONFIG['weight_decay'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['max_epochs'])
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=CONFIG['warmup_epochs'])
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val_f1 = -1.0
    best_state = None
    history = []
    start_epoch = 1

    # Check for existing checkpoint to resume training
    checkpoint_path = fold_output_dir / 'best_model.pth'
    history_path = fold_output_dir / 'training_history.json'

    if checkpoint_path.exists() and history_path.exists():
        print(f"Resuming training for fold {fold_idx} from checkpoint: {checkpoint_path}")
        model.load_state_dict(torch.load(checkpoint_path))
        with open(history_path, 'r') as f:
            history = json.load(f)

        if history:
            start_epoch = history[-1]['epoch'] + 1
            best_val_f1 = max(h['val_subtype_macro_f1'] for h in history) # Recalculate best_val_f1
            print(f"Resuming from epoch {start_epoch}, previous best F1: {best_val_f1:.4f}")
        else:
            print("Checkpoint found but history is empty. Starting from epoch 1.")

    for epoch in range(start_epoch, CONFIG['max_epochs'] + 1):
        if epoch <= 5:
            set_requires_grad(model.cnn_backbone, False)
            set_requires_grad(model.vit_backbone, False)
            set_requires_grad(model.cnn_proj, True)
            set_requires_grad(model.vit_proj, True)
            set_requires_grad(model.fusion, True)
            set_requires_grad(model.binary_head, True)
            set_requires_grad(model.subtype_head, True)
        else:
            set_requires_grad(model.cnn_backbone, True)
            set_requires_grad(model.vit_backbone, True)
            set_requires_grad(model.cnn_proj, True)
            set_requires_grad(model.vit_proj, True)
            set_requires_grad(model.fusion, True)
            set_requires_grad(model.binary_head, True)
            set_requires_grad(model.subtype_head, True)

        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, criterion, device)
        val_metrics = evaluate_model(model, val_loader, device)
        current_epoch_history = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'val_subtype_macro_f1': val_metrics['subtype_macro_f1'],
        }
        history.append(current_epoch_history)

        # Save training history after each epoch
        with open(fold_output_dir / 'training_history.json', 'w') as f:
            json.dump(history, f, indent=2)

        if val_metrics['subtype_macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['subtype_macro_f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            # Save best model state
            torch.save(best_state, fold_output_dir / 'best_model.pth')

        if epoch >= CONFIG['warmup_epochs']:
            scheduler.step()

        if epoch > CONFIG['early_stopping_patience'] and all(h['val_subtype_macro_f1'] <= best_val_f1 for h in history[-CONFIG['early_stopping_patience']:]):
            print(f"Early stopping triggered at epoch {epoch} due to no improvement in val_subtype_macro_f1.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate_model(model, test_loader, device)

    # Convert numpy arrays in test_metrics to lists for JSON serialization
    for key, value in test_metrics.items():
        if isinstance(value, np.ndarray):
            test_metrics[key] = value.tolist()

    # Save final test metrics
    with open(fold_output_dir / 'test_metrics.json', 'w') as f:
        json.dump(test_metrics, f, indent=2)

    return model, test_metrics, history


## 8. Run Training Across the 5 Patient-Disjoint Folds


In [ ]:
all_fold_results = []
for fold_idx in range(CONFIG['n_splits']):
    print(f'\n=== Starting fold {fold_idx} ===')
    model, test_metrics, history = run_fold(fold_idx, metadata)
    fold_record = {
        'fold_idx': fold_idx,
        'history': history,
        'test_metrics': test_metrics,
    }
    all_fold_results.append(fold_record)
    print(f'Fold {fold_idx} complete. subtype_macro_f1={test_metrics["subtype_macro_f1"]:.4f}')


=== Starting fold 0 ===
Resuming training for fold 0 from checkpoint: /content/drive/MyDrive/output_v3/fold_0/best_model.pth
Resuming from epoch 13, previous best F1: 0.1613


  0%|          | 0/196 [00:00<?, ?it/s]

Early stopping triggered at epoch 13 due to no improvement in val_subtype_macro_f1.
Fold 0 complete. subtype_macro_f1=0.1973

=== Starting fold 1 ===


  0%|          | 0/208 [00:00<?, ?it/s]

  0%|          | 0/208 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8.1 Diagnostic Audit for Fold 0

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Assuming all_fold_results[0] contains the data for Fold 0
fold_0_record = all_fold_results[0]
fold_idx = fold_0_record['fold_idx']
history = fold_0_record['history']
test_metrics = fold_0_record['test_metrics']

print(f"### Fold {fold_idx} - Summary")
print(f"Final Test subtype_macro_f1: {test_metrics['subtype_macro_f1']:.4f}")
print(f"Final Test binary_macro_f1: {test_metrics['binary_macro_f1']:.4f}")

print("\n--- Audit: Training and Validation Data --- ")
# Recreate train_df, val_df, test_df for Fold 0 for detailed analysis
train_df_0, test_df_0 = build_fold_split(fold_idx, metadata)

# Recreate validation split (same logic as in run_fold)
train_patients_0 = set(train_df_0['patient_id'])
remaining_0 = metadata[~metadata['patient_id'].isin(train_patients_0)].copy()
val_patients_0 = remaining_0['patient_id'].drop_duplicates().sample(max(1, int(0.1 * len(train_patients_0))), random_state=CONFIG['seed'])
val_df_0 = metadata[metadata['patient_id'].isin(val_patients_0)].copy()
train_df_0_final = train_df_0[~train_df_0['patient_id'].isin(val_patients_0)].copy()

print(f"Train samples: {len(train_df_0_final)}")
print(f"Validation samples: {len(val_df_0)}")
print(f"Test samples: {len(test_df_0)}")

print(f"Unique patients in Train: {train_df_0_final['patient_id'].nunique()}")
print(f"Unique patients in Validation: {val_df_0['patient_id'].nunique()}")
print(f"Unique patients in Test: {test_df_0['patient_id'].nunique()}")

print("\n--- Audit: Class Distributions (Fold 0) ---")
print("\nTrain Binary Label Distribution:")
print(train_df_0_final['binary_label'].value_counts(normalize=True))
print("\nTrain Subtype Label Distribution:")
print(train_df_0_final['subtype'].value_counts(normalize=True))

print("\nValidation Binary Label Distribution:")
print(val_df_0['binary_label'].value_counts(normalize=True))
print("\nValidation Subtype Label Distribution:")
print(val_df_0['subtype'].value_counts(normalize=True))

print("\nTest Binary Label Distribution:")
print(test_df_0['binary_label'].value_counts(normalize=True))
print("\nTest Subtype Label Distribution:")
print(test_df_0['subtype'].value_counts(normalize=True))

### Fold 0 - Summary
Final Test subtype_macro_f1: 0.1973
Final Test binary_macro_f1: 0.7451

--- Audit: Training and Validation Data --- 
Train samples: 6265
Validation samples: 454
Test samples: 1644
Unique patients in Train: 64
Unique patients in Validation: 6
Unique patients in Test: 17

--- Audit: Class Distributions (Fold 0) ---

Train Binary Label Distribution:
binary_label
1    0.719234
0    0.280766
Name: proportion, dtype: float64

Train Subtype Label Distribution:
subtype
DC    0.459218
MC    0.116520
F     0.109178
LC    0.080287
PT    0.072306
PC    0.063208
TA    0.059218
A     0.040064
Name: proportion, dtype: float64

Validation Binary Label Distribution:
binary_label
0    0.592511
1    0.407489
Name: proportion, dtype: float64

Validation Subtype Label Distribution:
subtype
F     0.312775
DC    0.270925
TA    0.145374
MC    0.136564
A     0.134361
Name: proportion, dtype: float64

Test Binary Label Distribution:
binary_label
1    0.561436
0    0.438564
Name: proportion,

### Training History Plots

In [ ]:
epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_f1s = [h['val_subtype_macro_f1'] for h in history]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss')
plt.title('Train Loss per Epoch (Fold 0)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_f1s, label='Validation Macro F1')
plt.title('Validation Subtype Macro F1 per Epoch (Fold 0)')
plt.xlabel('Epoch')
plt.ylabel('Macro F1 Score')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

### Test Metrics (Fold 0)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

print("\n--- Audit: Binary Classification Metrics ---")
print(f"Accuracy: {test_metrics['binary_accuracy']:.4f}")
print(f"Macro F1: {test_metrics['binary_macro_f1']:.4f}")
print(f"Balanced Accuracy: {test_metrics['binary_balanced_accuracy']:.4f}")
print(f"MCC: {test_metrics['binary_mcc']:.4f}")

print("\n--- Audit: Subtype Classification Metrics ---")
print(f"Accuracy: {test_metrics['subtype_accuracy']:.4f}")
print(f"Macro F1: {test_metrics['subtype_macro_f1']:.4f}")
print(f"Weighted F1: {test_metrics['subtype_weighted_f1']:.4f}")
print(f"Balanced Accuracy: {test_metrics['subtype_balanced_accuracy']:.4f}")
print(f"MCC: {test_metrics['subtype_mcc']:.4f}")

print("\n--- Audit: Per-class Subtype F1 ---")
# Extract using the exact keys saved from evaluate_model
subtype_true_labels = np.array(test_metrics['subtype_true'])
subtype_preds_labels = np.array(test_metrics['subtype_preds'])

per_class_f1 = f1_score(subtype_true_labels, subtype_preds_labels, average=None)
for i, f1 in enumerate(per_class_f1):
    print(f"  {CONFIG['subtype_order'][i]}: {f1:.4f}")

print("\n--- Audit: Prediction Class Distribution (Subtype) ---")
pred_subtype_counts = pd.Series(subtype_preds_labels).value_counts().sort_index()
pred_subtype_dist = pred_subtype_counts / len(subtype_preds_labels)
for i, count in pred_subtype_counts.items():
    print(f"  {CONFIG['subtype_order'][i]}: {count} ({pred_subtype_dist.loc[i]:.4f})")


--- Audit: Binary Classification Metrics ---
Accuracy: 0.7464
Macro F1: 0.7451
Balanced Accuracy: 0.7489
MCC: 0.4941

--- Audit: Subtype Classification Metrics ---
Accuracy: 0.2725
Macro F1: 0.1973
Weighted F1: 0.2746
Balanced Accuracy: 0.2600
MCC: 0.1770

--- Audit: Per-class Subtype F1 ---


KeyError: 'subtype_true'

### Confusion Matrices (Fold 0)

In [ ]:
from pathlib import Path

# Ensure output directory exists for plots
fold_0_output_dir = OUTPUT_DIR / f'fold_{fold_idx}'
fold_0_output_dir.mkdir(parents=True, exist_ok=True)

# Assuming confusion matrices are lists (due to JSON serialization fix)
cm_binary_display = np.array(test_metrics['binary_confusion_matrix'])
cm_subtype_display = np.array(test_metrics['subtype_confusion_matrix'])

plot_confusion_matrix(cm_binary_display, CONFIG['binary_classes'], f'Fold {fold_idx} Binary Confusion Matrix', str(fold_0_output_dir / 'confusion_binary.png'))
plot_confusion_matrix(cm_subtype_display, CONFIG['subtype_order'], f'Fold {fold_idx} Subtype Confusion Matrix', str(fold_0_output_dir / 'confusion_subtype.png'))

# Display the plots in the notebook
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.heatmap(cm_binary_display, annot=True, fmt='d', cmap='Blues', xticklabels=CONFIG['binary_classes'], yticklabels=CONFIG['binary_classes'])
plt.title(f'Fold {fold_idx} Binary Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.subplot(1, 2, 2)
sns.heatmap(cm_subtype_display, annot=True, fmt='d', cmap='Blues', xticklabels=CONFIG['subtype_order'], yticklabels=CONFIG['subtype_order'])
plt.title(f'Fold {fold_idx} Subtype Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.tight_layout()
plt.show()

NameError: name 'plot_confusion_matrix' is not defined

## Audit Findings and Recommendations

Based on the code inspection and the diagnostic output (which you should review after running the cells above), here's an assessment of the checklist items:

1.  **Dataset parsing and subtype labels:** The `parse_breakhis_filename` function and the `CONFIG` mappings appear correct and robust for the expected BreaKHis format.
2.  **Benign/malignant labels:** The binary mapping logic in `CONFIG` and parsing in `parse_breakhis_filename` are correctly implemented.
3.  **Patient/specimen IDs and StratifiedGroupKFold splitting:** The extraction of `patient_id` is correct, and `StratifiedGroupKFold` is used appropriately to ensure patient-disjoint splits while maintaining subtype distribution. The printed unique patient counts in train/val/test splits confirm this.
4.  **Fold 0 class distribution:** The diagnostic output will show the distribution. Often, severe class imbalance can lead to low macro F1 scores, especially for minority classes.
5.  **Training vs validation sample counts:** The diagnostic output will confirm the split sizes.
6.  **EfficientNet-B0 feature extraction:** Standard `timm` usage, seems correct.
7.  **ViT-Tiny feature extraction:** Standard `timm` usage, extracting the CLS token, seems correct.
8.  **Feature projection dimensions:** The `cnn_proj`, `vit_proj` and `fusion` layer dimensions are consistent with `CONFIG['fusion_dim'] = 256` and `CONFIG['magnification_embedding_dim'] = 64`.
9.  **Magnification encoding and adaptive fusion:** The `MagnificationAwareFusion` module's architecture correctly incorporates magnification embeddings and computes an alpha gate for fusion. This mechanism appears correctly implemented.
10. **Hierarchical binary/subtype heads:** The `binary_head` and `subtype_head` are simple linear layers matching the expected output class counts.
11. **Binary, subtype, and consistency loss implementation:** The `HierarchicalLoss` combines `CrossEntropyLoss` for binary and subtype, and `KLDivLoss` for consistency. The logic seems correct according to the design.
12. **Class weights/sampling:** `compute_class_weights` correctly calculates inverse frequency weights. These weights are applied to the `CrossEntropyLoss` during training. While a `WeightedRandomSampler` is not used in the `DataLoader`, the `CrossEntropyLoss` weights are a common strategy for imbalance.
13. **Optimizer, learning rates, scheduler and freezing/unfreezing:** `AdamW` with differential learning rates, `CosineAnnealingLR` with `LinearLR` warmup, and progressive unfreezing (backbones frozen for first 5 epochs) are well-defined strategies and correctly implemented.
14. **Checkpoint save/resume logic:** The checkpointing mechanism for saving `best_model.pth` and `training_history.json`, and resuming from them, is robustly implemented. `start_epoch` and `best_val_f1` are correctly restored or initialized.
15. **Early-stopping logic:** The condition `epoch > CONFIG['early_stopping_patience'] and all(h['val_subtype_macro_f1'] <= best_val_f1 for h in history[-CONFIG['early_stopping_patience']:])` is a standard and correct way to implement patience-based early stopping.
16. **Validation metric calculation, especially macro-F1:** `evaluate_model` correctly calculates macro F1 using `sklearn.metrics.f1_score(average='macro')`.
17. **Any stale checkpoint being incorrectly resumed:** The logic correctly checks for existing history and adjusts `start_epoch` and `best_val_f1` accordingly, preventing stale checkpoints from leading to incorrect resumption.

### Overall Assessment of Low F1:

Based on the implementation review, there are **no obvious bugs in the code** that would definitively explain the low `subtype_macro_f1` of 0.1973 for Fold 0. The model architecture, training loop, loss functions, and evaluation metrics all appear to be implemented as described in the design.

The likely reasons for the low F1 and early stopping are:

*   **Severe Class Imbalance:** The subtype distribution, especially if some classes have very few samples, can make it extremely challenging for the model to learn meaningful representations for those minority classes, leading to low F1 scores for them and thus a low overall macro F1.
*   **Model Capacity/Complexity:** While EfficientNet-B0 and ViT-Tiny are capable models, the complexity of the BreaKHis dataset with fine-grained subtype classification might require a larger model, more training data, or a more sophisticated training regimen (e.g., more advanced augmentation, different optimizer/scheduler settings).
*   **Hyperparameter Tuning:** The current hyperparameters (learning rates, weight decay, early stopping patience, etc.) might not be optimal for achieving higher performance on this specific task, even if the general approach is sound.
*   **Short Training:** Early stopping at epoch 13 (out of 50 max epochs) suggests that the validation F1 either converged very quickly or started to degrade, potentially indicating the model struggles to learn further with the current settings or has already overfit to the training data to some extent. The training history plots will be crucial here.

### Next Steps and Recommendation:

**Minimum Necessary Correction (Prioritize Data Understanding):**

1.  **Analyze the Class Distributions:** Pay very close attention to the class distributions printed by the diagnostic cells. If there are subtypes with very few samples in the training set (e.g., < 50 samples), the model will struggle to learn them effectively.
2.  **Examine Per-Class F1:** The per-class F1 scores will show which specific subtypes the model is failing on. This will help confirm if minority classes are the issue.
3.  **Review Training History Plots:** Look for signs of overfitting (train loss decreases, val F1 stagnates or increases then drops) or underfitting (both train loss and val F1 are stagnant or poor).

**Should Fold 0 be rerun?**

**Yes, Fold 0 should be rerun** if you identify any concrete issues from the diagnostic output (e.g., extremely skewed class distributions that point to a data error, or an obvious misconfiguration). If the diagnostic output simply confirms class imbalance or a model struggling, but no outright bug, then rerunning Fold 0 *after considering potential solutions* (like adjustments to class weighting, adding a `WeightedRandomSampler` if `CrossEntropyLoss` weights aren't enough, or exploring data augmentation strategies) would be beneficial. For now, since the code appears correct, it's about optimizing performance, not fixing a crash.

After you have reviewed the diagnostic outputs, we can discuss potential next steps for improving performance, such as:
*   More aggressive data augmentation, especially for minority classes.
*   Exploring a `WeightedRandomSampler` in the DataLoader.
*   Adjusting hyperparameters like learning rates, weight decay, or early stopping patience.
*   Considering a larger model or more complex fusion mechanism if the current model's capacity is insufficient.


## 9. Evaluation and Aggregated Results


In [ ]:
def aggregate_results(results):
    metrics = [r['test_metrics'] for r in results]
    summary = {}
    for key in [
        'binary_accuracy', 'binary_macro_f1', 'binary_balanced_accuracy', 'binary_mcc',
        'subtype_accuracy', 'subtype_macro_f1', 'subtype_weighted_f1', 'subtype_balanced_accuracy', 'subtype_mcc'
    ]:
        # Ensure we handle both numpy types and already converted lists from json
        values = []
        for m in metrics:
            val = m[key]
            if isinstance(val, list):
                values.append(float(val[0]) if len(val) == 1 else val) # Handle potential single-element list from json conversion
            else:
                values.append(float(val))

        values = np.array(values)
        summary[key] = {
            'mean': float(values.mean()),
            'std': float(values.std()),
            'min': float(values.min()),
            'max': float(values.max()),
        }

    # Handle confusion matrices separately as they are already lists
    if metrics and 'binary_confusion_matrix' in metrics[0]:
        summary['binary_confusion_matrices'] = [m['binary_confusion_matrix'] for m in metrics]
    if metrics and 'subtype_confusion_matrix' in metrics[0]:
        summary['subtype_confusion_matrices'] = [m['subtype_confusion_matrix'] for m in metrics]

    return summary

aggregated = aggregate_results(all_fold_results)
print(json.dumps(aggregated, indent=2))

{
  "binary_accuracy": {
    "mean": 0.7463503649635036,
    "std": 0.0,
    "min": 0.7463503649635036,
    "max": 0.7463503649635036
  },
  "binary_macro_f1": {
    "mean": 0.7450591231763521,
    "std": 0.0,
    "min": 0.7450591231763521,
    "max": 0.7450591231763521
  },
  "binary_balanced_accuracy": {
    "mean": 0.7489124440444008,
    "std": 0.0,
    "min": 0.7489124440444008,
    "max": 0.7489124440444008
  },
  "binary_mcc": {
    "mean": 0.4941463108982971,
    "std": 0.0,
    "min": 0.4941463108982971,
    "max": 0.4941463108982971
  },
  "subtype_accuracy": {
    "mean": 0.2725060827250608,
    "std": 0.0,
    "min": 0.2725060827250608,
    "max": 0.2725060827250608
  },
  "subtype_macro_f1": {
    "mean": 0.19728322018251812,
    "std": 0.0,
    "min": 0.19728322018251812,
    "max": 0.19728322018251812
  },
  "subtype_weighted_f1": {
    "mean": 0.27460765285394917,
    "std": 0.0,
    "min": 0.27460765285394917,
    "max": 0.27460765285394917
  },
  "subtype_balanced_acc


## 10. Confusion Matrices


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def plot_confusion_matrix(cm, labels, title, fname):
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    # plt.close() # Removed to allow plots to display in notebook if the cell is run directly

for fold_idx, entry in enumerate(all_fold_results):
    # Ensure the output directory for this fold exists
    fold_output_dir = OUTPUT_DIR / f'fold_{fold_idx}'
    fold_output_dir.mkdir(parents=True, exist_ok=True)

    cm_binary = entry['test_metrics']['binary_confusion_matrix']
    cm_subtype = entry['test_metrics']['subtype_confusion_matrix']
    plot_confusion_matrix(cm_binary, ['Benign', 'Malignant'], f'Fold {fold_idx} Binary Confusion Matrix', str(fold_output_dir / 'confusion_binary.png'))
    plot_confusion_matrix(cm_subtype, CONFIG['subtype_order'], f'Fold {fold_idx} Subtype Confusion Matrix', str(fold_output_dir / 'confusion_subtype.png'))


## 11. ROC Curves


In [ ]:

def compute_roc_curve(y_true, y_score, pos_label=1):
    from sklearn.metrics import roc_curve, auc
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=pos_label)
    area = auc(fpr, tpr)
    return fpr, tpr, area

for fold_idx, entry in enumerate(all_fold_results):
    subtype_true = torch.cat([batch['subtype_label'] for batch in iter(DataLoader(BreakHisDataset(pd.DataFrame(), 'test', build_transforms('val')), batch_size=CONFIG['batch_size']))]) if False else None



## 12. Fusion Gate Analysis


In [ ]:

def gate_analysis(model, loader, device):
    model.eval()
    alpha_by_mag = {m: [] for m in CONFIG['magnification_levels']}
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            magnification_idx = batch['magnification_index'].to(device)
            outputs = model(images, magnification_idx)
            alpha = outputs['alpha'].detach().cpu().flatten()
            for idx, mag in enumerate(CONFIG['magnification_levels']):
                mask = (batch['magnification_index'].numpy() == idx)
                if np.any(mask):
                    alpha_by_mag[mag].extend(alpha[mask].numpy().tolist())
    for mag in CONFIG['magnification_levels']:
        print(f'Magnification {mag}: mean alpha={np.mean(alpha_by_mag[mag]):.4f}, std={np.std(alpha_by_mag[mag]):.4f}')
    return alpha_by_mag



## 13. Grad-CAM + Attention + t-SNE


In [ ]:

def compute_tsne(features, labels):
    tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=30)
    coords = tsne.fit_transform(features)
    plt.figure(figsize=(8, 6))
    for idx in range(8):
        sel = labels == idx
        plt.scatter(coords[sel, 0], coords[sel, 1], s=20, label=CONFIG['subtype_order'][idx], alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title('t-SNE embedding of fused features')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/OMNet/v3_outputs/tsne_embeddings.png', dpi=200)
    plt.close()
    return coords



## 14. Error Analysis and Export


In [ ]:

def patient_score_metric(y_true, y_pred):
    return np.mean(y_true == y_pred)

def export_results(results):
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    for idx in range(CONFIG['n_splits']):
        os.makedirs(f"{CONFIG['output_dir']}/fold_{idx}", exist_ok=True)

    with open(f"{CONFIG['output_dir']}/aggregated_results.json", 'w') as f:
        json.dump(aggregate_results(results), f, indent=2)

    with open(f"{CONFIG['output_dir']}/experiment_config.json", 'w') as f:
        json.dump(CONFIG, f, indent=2)

    print(f'Export complete to {CONFIG["output_dir"]}')

export_results(all_fold_results)


AttributeError: 'list' object has no attribute 'tolist'